#Importing Functions 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Silver Data Transformation

In [0]:
path='abfss://bronzenetflix@naiyarstorage.dfs.core.windows.net/netflix_titles'
df=spark.read.format('delta')\
    .option('header',True)\
        .option('inferSchema',True)\
            .load(path)

In [0]:
df.printSchema()

# type casting

In [0]:

df = df.withColumn(
    "duration_minutes",
    expr("try_cast(duration_minutes as int)")
).withColumn(
    "duration_seasons",
    expr("try_cast(duration_seasons as int)")
)

### Handling Null Values
- df.fillna(0) - every null values will be replaced by 0
- df.fillna(0,subset=['durations_minutes','duration_seasons']) 
- df.fillna({'durations_minutes : 0 '," duration_seasons : 0 "})

In [0]:
 df=df.fillna({"duration_minutes":0 ,"duration_seasons":1})
 

In [0]:
df.limit(50).display()

In [0]:
df=df.withColumn('short_title',split(col("title"),':')[0])\
    .withColumn('rating', split(col("rating"),'-')[0])


In [0]:
df.display()

In [0]:
df=df.withColumn('type_flag',when(col("type")=='Movie',1)\
    .when(col('type')=='TV Show',2)\
        .otherwise(0))

In [0]:
df.display()

In [0]:
from pyspark.sql.window import Window

In [0]:
windowspec=Window.orderBy(col('duration_minutes').desc())
df=df.withColumn('duration_ranking',dense_rank().over(windowspec))

In [0]:
df.display()

In [0]:
df_vis=df.groupBy("type").agg(count("*").alias('total_count'))
df.display()

In [0]:
silver_path='abfss://silvernetflix@naiyarstorage.dfs.core.windows.net/netflix_titles'
df.write.format('delta')\
    .mode('overwrite')\
        .option('path',silver_path)